# Tutorial 2: Stop detection in trajectories

This notebook shows how to process device-level trajectory data, in different formats, to detect stops using ```nomad```. Stop detection is an important step in
pre-processing trajectory data and in making sense of trajectories by grouping together pings that reflect stationary behavior. The output of stop-detection algorithms is commonly a "stop table", indicating when a stop started, its duration, and a pair of coordinates that approximates the location of the group of pings (typically the centroid). Alternatively, ```nomad``` allows users to retrieve a cluster label for each ping (useful for plotting, for example)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

In [ ]:
import filters
import daphmeIO as loader
import constants as constants
import stop_detection_modified as SD

## Load data sample

For these examples we load some test data from ```nomad``` which has the following trajectory columns (notice that ```traj_cols``` maps the data's column names to default names used by ```nomad```)

In [ ]:
traj_cols = {'user_id':'user_id',
             'latitude':'dev_lat',
             'longitude':'dev_lon',
             'datetime':'local_datetime'}

In [ ]:
path = '../data/sample4/'
data = loader.from_file(path, traj_cols=traj_cols, format='csv')

This synthetic data has records for 100 users for a 1 week period, with spherical coordinates (lat, lon) and datetime format for the time component of each ping. 

In [ ]:
data.head()

## Stop detection algorithms

The stop detection algorithms in ```nomad``` are applied to each user's trajectories separately. Thus, we demonstrate first by sampling a single user's data

In [ ]:
user_sample = data.loc[data.user_id == "angry_spence"]

### Sequential stop detection (Lachesis)

The data could also come with different formats for spatial and temporal variables:

In [ ]:
# Longitude, Latitude, timestamp (unix time)
traj_cols_lon_lat_timestamp = {'user_id':'user_id',
                               'latitude':'dev_lat',
                               'longitude':'dev_lon',
                               'timestamp':'unix_time'}

user_sample4_lon_lat_timestamp = user_sample4.copy()
user_sample4_lon_lat_timestamp['unix_time'] = user_sample4_lon_lat_timestamp['local_datetime'].astype('int64') // 10**9

# x, y, datetime
traj_cols_x_y_datetime = {'user_id':'user_id',
                          'x':'x',
                          'y':'y',
                          'datetime':'local_datetime'}

user_sample4_x_y_datetime = user_sample4.copy()
user_sample4_x_y_datetime = filters.to_projection(user_sample4_x_y_datetime, longitude='dev_lon', latitude='dev_lat')

# x, y, timestamp (unix time)
traj_cols_x_y_timestamp = {'user_id':'user_id',
                          'x':'x',
                          'y':'y',
                          'timestamp':'unix_time'}

user_sample4_x_y_timestamp = user_sample4.copy()
user_sample4_x_y_timestamp = filters.to_projection(user_sample4_x_y_timestamp, longitude='dev_lon', latitude='dev_lat')
user_sample4_x_y_timestamp['unix_time'] = user_sample4_x_y_timestamp['local_datetime'].astype('int64') // 10**9

## Stop Detection Algorithms

### Lachesis

We need to pass some reasonable parameters for:
* ```dur_min```: Minimum duration for a stay in minutes.
* ```dt_max```: Maximum duration permitted between consecutive pings in a stay in minutes (dt_max should be greater than dur_min).
* ```delta_roam```: Maximum roaming distance for a stay in meters.

In [ ]:
DUR_MIN = 60
DT_MAX = 120
DELTA_ROAM = 50

Minimum stop table output columns:

In [ ]:
SD.lachesis(user_sample4, DUR_MIN, DT_MAX, DELTA_ROAM, traj_cols=traj_cols, complete_output=False)

All stop table output columns:

In [ ]:
SD.lachesis(user_sample4, DUR_MIN, DT_MAX, DELTA_ROAM, traj_cols=traj_cols, complete_output=True)

Lachesis with longitude, latitude, and timestamp (unix time):

In [ ]:
SD.lachesis(user_sample4_lon_lat_timestamp, DUR_MIN, DT_MAX, DELTA_ROAM, traj_cols=traj_cols_lon_lat_timestamp, complete_output=False)

Lachesis with x, y, and timestamp (unix time):

In [ ]:
SD.lachesis(user_sample4_x_y_timestamp, DUR_MIN, DT_MAX, DELTA_ROAM, traj_cols=traj_cols_x_y_timestamp, complete_output=False)

Lachesis with x, y, and datetime:

In [ ]:
SD.lachesis(user_sample4_x_y_datetime, DUR_MIN, DT_MAX, DELTA_ROAM, traj_cols=traj_cols_x_y_datetime, complete_output=False)

### Temporal DBSCAN

We need to pass some reasonable parameters for:
* ```time_thresh```: Time threshold in minutes for identifying neighbors.
* ```dist_thresh```: Distance threshold in meters for identifying neighbors.
* ```min_pts```: Minimum number of points required to form a dense region (core point).

In [ ]:
TIME_THRESH = 100
DIST_THRESH = 40
MIN_PTS = 10

We can get the final cluster and core labels for each of the data points:

In [ ]:
SD._temporal_dbscan_labels(user_sample4, TIME_THRESH, DIST_THRESH, MIN_PTS, complete_output=False, traj_cols=traj_cols)

Or we can also get a stop table:

In [ ]:
SD.temporal_dbscan(user_sample4, TIME_THRESH, DIST_THRESH, MIN_PTS, complete_output=False, traj_cols=traj_cols)

Temporal DBSCAN with longitude, latitude, and timestamp (unix time):

In [ ]:
SD.temporal_dbscan(user_sample4_lon_lat_timestamp, TIME_THRESH, DIST_THRESH, MIN_PTS, complete_output=False, traj_cols=traj_cols_lon_lat_timestamp)

Temporal DBSCAN with x, y, and timestamp (unix time):

In [ ]:
SD.temporal_dbscan(user_sample4_x_y_timestamp, TIME_THRESH, DIST_THRESH, MIN_PTS, complete_output=False, traj_cols=traj_cols_x_y_timestamp)

Temporal DBSCAN with x, y, and datetime:

In [ ]:
SD.temporal_dbscan(user_sample4_x_y_datetime, TIME_THRESH, DIST_THRESH, MIN_PTS, complete_output=False, traj_cols=traj_cols_x_y_datetime)